In [ ]:
# ⛳ Mount Drive and set variables
#!fusermount -u /content/drive

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

GITHUB_USER = "saeedzns"          # <<< EDIT ME
REPO_NAME   = "Ego4d-LiteSTA"             # <<< default as requested
REPO_SSH    = f"git@github.com:{GITHUB_USER}/{REPO_NAME}.git"
WORKDIR     = f"/content/drive/MyDrive/{REPO_NAME}"
SSH_DIR     = "/content/drive/MyDrive/.ssh"

print("Repo:", REPO_SSH)
print("Workdir:", WORKDIR)


# 🔐 SSH + git config + clone/pull
# 1) Git identity (optional but recommended)
!git config --global user.name "saeedzns"
!git config --global user.email "zns1992@gmail.com"

In [ ]:
# Git keepalive + sane pack settings for Colab
import os, subprocess
os.environ['GIT_SSH_COMMAND'] = 'ssh -o ServerAliveInterval=30 -o ServerAliveCountMax=10'
subprocess.run(['git','config','--global','core.compression','9'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(['git','config','--global','pack.threads','2'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print('Configured GIT_SSH_COMMAND and git pack settings')


In [ ]:
# 2) SSH agent and key from Drive (key filename: colab_egolite)
%%bash -s "$SSH_DIR"
SSH_DIR="$1"
mkdir -p "$SSH_DIR"
chmod 700 "$SSH_DIR"
eval "$(ssh-agent -s)"
ssh-add "$SSH_DIR/colab_egolite" >/dev/null 2>&1 || true
mkdir -p ~/.ssh && chmod 700 ~/.ssh
cat <<EOF > ~/.ssh/config
Host github.com
  HostName github.com
  User git
  IdentityFile $SSH_DIR/colab_egolite
  IdentitiesOnly yes
StrictHostKeyChecking accept-new
EOF
chmod 600 ~/.ssh/config
ssh-keyscan -t rsa,ed25519 github.com >> ~/.ssh/known_hosts 2>/dev/null || true
chmod 644 ~/.ssh/known_hosts

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

In [ ]:
import psutil

ram_gb = psutil.virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

# Videomae pretraining

In [ ]:
# ==== Ego-only VideoMAE pretraining from scratch on precomputed tensors ====
# Assumes you already uploaded:
#   /content/drive/MyDrive/Ego4d_STA/videomae/manifests/ego4d_sta_clips.jsonl
#   /content/drive/MyDrive/Ego4d_STA/videomae/tensors/<uid>.pt
# Tensors are shape (3, 16, 224, 224), float32 in [0,1].

import os
import json
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from google.colab import drive
from transformers import VideoMAEConfig, VideoMAEForPreTraining

# ---------- Paths / basic config ----------
DRIVE_ROOT = "/content/drive/MyDrive"
VIDEOMAE_ROOT = f"{DRIVE_ROOT}/Ego4d_STA/videomae"
MANIFEST_PATH = f"{VIDEOMAE_ROOT}/manifests/ego4d_sta_clips.jsonl"
CHECKPOINT_DIR = f"{VIDEOMAE_ROOT}/checkpoints_ego_scratch"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Training hyperparameters (tune if needed)
BATCH_SIZE = 16        # reduce if you get OOM (e.g., 8 or 4)
NUM_EPOCHS = 25        # bump if you have more GPU time
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.05
MASK_RATIO = 0.90      # like VideoMAE paper
GRAD_ACCUM_STEPS = 4   # increase if you want larger effective batch

# Optional: limit data for quick runs/debug
MAX_SAMPLES = None     # e.g. 500 for quick debug, or None for all 2323

# ---------- 1) Load manifest ----------
records = []
with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        # rec["tensor_path"] should already be an absolute Drive path
        records.append(rec)

if MAX_SAMPLES is not None:
    records = records[:MAX_SAMPLES]

print(f"[Info] Loaded {len(records)} records from manifest: {MANIFEST_PATH}")


# ---------- 2) Dataset using precomputed tensors ----------
class EgoTensorDataset(Dataset):
    def __init__(self, recs):
        self.recs = recs
        # VideoMAE standard normalization
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1, 1)

    def __len__(self):
        return len(self.recs)

    def __getitem__(self, idx):
        rec = self.recs[idx]
        tpath = rec["tensor_path"]
        x = torch.load(tpath)  # (3, 16, 224, 224), float32, [0,1]
        # Normalize as in ImageNet / VideoMAE
        x = (x - self.mean) / self.std
        # Stored as (C, T, H, W) -> model expects (B, T, C, H, W)
        x = x.permute(1, 0, 2, 3)  # (T, C, H, W)
        return {"pixel_values": x}


dataset = EgoTensorDataset(records)
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

print(f"[Info] Dataset size: {len(dataset)} clips, batch_size={BATCH_SIZE}")


# ---------- 3) Build VideoMAE model from scratch ----------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Info] Using device: {device}")

# Use base config for architecture, but DO NOT load pretrained weights
base_name = "MCG-NJU/videomae-base"
config = VideoMAEConfig.from_pretrained(base_name)

# Make sure config matches your tensors: (3,16,224,224)
config.num_frames = 16
config.image_size = 224

model = VideoMAEForPreTraining(config)
model.to(device)

# Compute number of patches for masking
patch_size = config.patch_size          # typically 16
tubelet_size = config.tubelet_size      # typically 2
num_frames = config.num_frames          # 16
img_size = config.image_size            # 224

t_tokens = num_frames // tubelet_size
h_tokens = img_size // patch_size
w_tokens = img_size // patch_size
NUM_PATCHES = t_tokens * h_tokens * w_tokens

print(f"[VideoMAE] num_patches for masking: {NUM_PATCHES} "
      f"(T_tokens={t_tokens}, H_tokens={h_tokens}, W_tokens={w_tokens})")

# ---------- 4) Optimizer / scaler ----------
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scaler = torch.amp.GradScaler()

# Optional resume
last_ckpt = os.path.join(CHECKPOINT_DIR, "videomae_ego_scratch_last.pt")
resumed = False
if os.path.exists(last_ckpt):
    print(f"[Info] Resuming from {last_ckpt}")
    ckpt = torch.load(last_ckpt, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scaler.load_state_dict(ckpt["scaler"])
    start_epoch = ckpt["epoch"] + 1
    resumed = True
else:
    start_epoch = 0

# ---------- 5) Training loop ----------
global_step = 0
model.train()
epoch_losses = []

for epoch in range(start_epoch, NUM_EPOCHS):
    running_loss = 0.0
    num_batches = 0
    pbar = tqdm(dataloader, desc=f"[Epoch {epoch+1}/{NUM_EPOCHS}]")

    for batch in pbar:
        pixel_values = batch["pixel_values"].to(device)  # (B, T, C, H, W)
        B = pixel_values.size(0)

        # Build random boolean mask of shape (B, NUM_PATCHES)
        num_mask = int(MASK_RATIO * NUM_PATCHES)
        bool_masked_pos = torch.zeros(B, NUM_PATCHES, dtype=torch.bool, device=device)
        idx = torch.rand(B, NUM_PATCHES, device=device).argsort(dim=1)[:, :num_mask]
        bool_masked_pos.scatter_(1, idx, True)

        with torch.amp.autocast(device_type="cuda" if device == "cuda" else "cpu"):
            outputs = model(pixel_values=pixel_values, bool_masked_pos=bool_masked_pos)
            loss = outputs.loss

        loss = loss / GRAD_ACCUM_STEPS
        scaler.scale(loss).backward()

        if (global_step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item() * GRAD_ACCUM_STEPS
        num_batches += 1
        global_step += 1

        pbar.set_postfix({"loss": running_loss / max(1, num_batches)})

    avg_loss = running_loss / max(1, num_batches)
    epoch_losses.append(float(avg_loss))
    print(f"[Epoch {epoch+1}] avg MAE loss = {avg_loss:.4f}")

    # Save epoch checkpoint (model + optimizer + scaler)
    ckpt_payload = {
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "config": config.to_dict(),
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "mask_ratio": MASK_RATIO,
        "grad_accum_steps": GRAD_ACCUM_STEPS,
        "num_patches": NUM_PATCHES,
        "dataset_size": len(dataset),
        "max_samples": MAX_SAMPLES,
    }

    ckpt_path = os.path.join(CHECKPOINT_DIR, f"videomae_ego_scratch_epoch{epoch+1}.pt")
    torch.save(ckpt_payload, ckpt_path)
    torch.save(ckpt_payload, last_ckpt)

    print(f"[Save] Epoch {epoch+1} checkpoint -> {ckpt_path}")
    print(f"[Save] Updated last checkpoint -> {last_ckpt}")

# ---------- 6) Save summary + configs as JSON ----------
summary = {
    "manifest_path": MANIFEST_PATH,
    "dataset_size": len(dataset),
    "max_samples": MAX_SAMPLES,
    "device": device,
    "resumed": resumed,
    "start_epoch": start_epoch,
    "num_epochs": NUM_EPOCHS,
    "final_epoch": NUM_EPOCHS - 1,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "mask_ratio": MASK_RATIO,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "num_patches": NUM_PATCHES,
    "t_tokens": t_tokens,
    "h_tokens": h_tokens,
    "w_tokens": w_tokens,
    "config": config.to_dict(),
    "epoch_losses": epoch_losses,
    "last_checkpoint": last_ckpt,
    "checkpoint_dir": CHECKPOINT_DIR,
}

summary_path = os.path.join(CHECKPOINT_DIR, "videomae_ego_scratch_summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("[Done] Ego-only VideoMAE pretraining finished.")
print(f"[Important] Use {last_ckpt} (or any epoch ckpt) later in your Track B video-encoder init.")
print(f"[Summary] Written summary JSON -> {summary_path}")
